# GW170817 PE with SHARPy + mlgw_bns_jax — Paper-Equivalent Setup (arXiv:2210.15684)

This notebook reproduces the parameter estimation (PE) setup described in **Section IV of arXiv:2210.15684** (the original `mlgw_bns` paper), using the **SHARPy SMC sampler** with the `mlgw_bns_jax` waveform monkey-patched in place of SHARPy's default IMRPhenomD template.

## Sampled parameters

All 13 parameters are sampled simultaneously:

| Index | Name | Description |
|:---:|---|---|
| 0 | `ra` | Right ascension |
| 1 | `dec` | Declination |
| 2 | `log_distance` | $\ln(d_L / \mathrm{Mpc})$ — log luminosity distance |
| 3 | `theta_jn` | Inclination angle |
| 4 | `phic` | Coalescence phase |
| 5 | `psi` | Polarisation angle |
| 6 | `chirp_mass` | Chirp mass $\mathcal{M}_c$ (M$_\odot$) |
| 7 | `mass_ratio` | Mass ratio $q = m_2/m_1 \in [0.5, 1]$ |
| 8 | `tc` | Coalescence time (s, relative to trigger) |
| 9 | `chi1` | Aligned spin of primary |
| 10 | `chi2` | Aligned spin of secondary |
| 11 | `lambda_1` | Tidal deformability of primary |
| 12 | `lambda_2` | Tidal deformability of secondary |

## Key differences from the naive SHARPy script

The following corrections have been applied to match the paper (compared to `gw170817_pe_sharpy_original.py` / `sharpy_mlgw_bns_jax_pe_full.ipynb`):

| Setting | Wrong (original script) | Correct (paper §IV) |
|---|---|---|
| Data duration | 4 s | **32 s** |
| $f_{\rm upper}$ | 2000 Hz | **2048 Hz** (Nyquist for 4096 Hz sampling) |
| Distance prior | Flat in $\ln d_L$ | **Uniform in volume** $p(d_L) \propto d_L^2$ |
| Inclination prior | Flat uniform | **Sine prior** $p(\theta) \propto \sin\theta$ |
| Declination prior | Flat uniform | **Cosine prior** $p(\delta) \propto \cos\delta$ |
| Phase factor | $e^{-i\phi_c}$ | $e^{-2i\phi_c}$ (LAL/Bilby convention) |
| Number of particles | 500 | **1000** (matching ~`nlive=1000` in paper's dynesty run) |

## 1 — Imports and JAX setup

In [ ]:
from __future__ import annotations
import os, sys, time
from functools import partial
import numpy as np

os.environ.setdefault("JAX_PLATFORMS", "cpu")
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
print("JAX devices:", jax.devices())

## 2 — Load waveform model

The `mlgw_bns_jax` model is loaded from the HDF5 weights file `mlgw_bns_jax_model.h5` using the `load_predict` helper and immediately JIT-compiled with JAX for fast repeated evaluations during sampling.

In [ ]:
sys.path.insert(0, os.getcwd())
from jax_import_n_predict import load_predict

MODEL_PATH = "mlgw_bns_jax_model.h5"
_mlgw_predict = jax.jit(load_predict(MODEL_PATH))
print("Model loaded and JIT-compiled.")

## 3 — Monkey-patch SHARPy

We replace SHARPy's internal `template` function (which normally calls IMRPhenomD via `ripplegw`) with our `mlgw_bns_jax` BNS waveform **without modifying any SHARPy source file**. The patch must be applied *before* importing `GWNetwork` and `log_likelihood_det`, because those symbols capture the template reference at import time.

**Critical correction vs. the naive script**: The phase factor is `exp(-2j * phic)` (not `exp(-1j * phic)`). This factor-of-2 difference matches the LAL/Bilby convention used throughout the paper (arXiv:2210.15684).

In [ ]:
import sharpy.GW_likelihood as _gw_mod
from sharpy.utils import McQ2Masses


def _template_mlgw_bns(params, frequency_array):
    """
    mlgw_bns_jax replacement for SHARPy's IMRPhenomD template.

    Parameter vector convention (13 parameters):
      [0] ra, [1] dec, [2] log(d_L/Mpc), [3] theta_jn, [4] phic,
      [5] psi,  [6] mc (M_sun), [7] q,  [8] tc (s, relative to trigger),
      [9] chi1, [10] chi2, [11] lambda_1, [12] lambda_2
    """
    mc          = params[6]
    q           = params[7]
    m1_msun, m2_msun = McQ2Masses(mc, q)
    total_mass  = m1_msun + m2_msun
    chi1        = params[9]
    chi2        = params[10]
    lambda_1    = params[11]
    lambda_2    = params[12]
    phic        = params[4]
    dist_mpc    = jnp.exp(params[2])
    inclination = params[3]

    mlgw_params = jnp.array([q, lambda_1, lambda_2, chi1, chi2])

    hp, hc = _mlgw_predict(
        mlgw_params, frequency_array,
        total_mass=total_mass,
        distance_mpc=dist_mpc,
        inclination=inclination,
    )

    # CORRECTION vs. naive script: phase factor = exp(-2j * phic)
    # This matches the LAL/Bilby convention used in the paper.
    phase_factor = jnp.exp(-2j * phic)
    hp = hp * phase_factor
    hc = hc * phase_factor

    return hp, hc


# Perform the monkey-patch BEFORE importing GWNetwork
_gw_mod.template = _template_mlgw_bns

from sharpy.GW_likelihood import GWNetwork, log_likelihood_det
from sharpy.smc_functions import run_sharpy

print("SHARPy monkey-patched successfully.")

## 4 — Event constants

Constants for GW170817, chosen to match Section IV of arXiv:2210.15684.

**Critical corrections vs. the naive script**:
- **Duration = 32 s** (not 4 s) — the paper analyses a 32 s segment around the event.
- **f_upper = 2048 Hz** (not 2000 Hz) — this is the Nyquist frequency for a 4096 Hz sampling rate.

In [ ]:
TRIGGER_TIME          = 1187008882.43    # GPS
SEGMENT_DURATION      = 32.0            # s — PAPER USES 32 s (not 4 s)
SAMPLING_RATE         = 4096            # Hz
F_LOWER               = 20.0            # Hz
F_UPPER               = 2048.0          # Hz — Nyquist (not 2000 Hz)
POST_TRIGGER_DURATION = 2               # s after trigger
DATA_START_GPS        = 1187008867      # GPS start of downloaded files
DATA_DIR              = "gw170817_data"
OUTDIR                = "outdir_GW170817_sharpy_paper"
LABEL                 = "GW170817_sharpy_paper"

os.makedirs(OUTDIR, exist_ok=True)
print(f"Analysis segment: {SEGMENT_DURATION} s, f_lower={F_LOWER} Hz, f_upper={F_UPPER} Hz")

## 5 — Load GW data

We load open GWOSC strain data for H1, L1, and V1 from the local `gw170817_data/` directory. The analysis segment starts at `trigger_time + post_trigger_duration - segment_duration`, i.e. 30 s before the trigger and ends 2 s after.

If the data files are not present, download them from [GWOSC](https://gwosc.org/eventapi/html/GWTC-1-confident/GW170817/).

In [ ]:
import glob

start_time = TRIGGER_TIME + POST_TRIGGER_DURATION - SEGMENT_DURATION
detector_names = ["H1", "L1", "V1"]
detector_settings = {}

for det in detector_names:
    matched = []
    for ext in ("txt", "hdf5", "gwf"):
        pattern = os.path.join(DATA_DIR, f"*{det}*GWOSC*.{ext}")
        matched = sorted(glob.glob(pattern))
        if matched:
            break
    if not matched:
        print(f"WARNING: No data for {det} found in {DATA_DIR}, skipping.")
        continue

    detector_settings[det] = {
        "data_file":     matched[0],
        "channel":       "GWOSC",
        "trigger_time":  TRIGGER_TIME,
        "duration":      SEGMENT_DURATION,
        "sampling_rate": SAMPLING_RATE,
        "f_lower":       F_LOWER,
        "f_upper":       F_UPPER,
        "psd_file":      None,
        "psd_method":    "welch",
        "download_data": False,
        "zero_noise":    False,
    }
    print(f"  {det}: {matched[0]}")

if not detector_settings:
    raise FileNotFoundError(
        f"No data files found in {DATA_DIR}. "
        "Download from https://gwosc.org/eventapi/html/GWTC-1-confident/GW170817/"
    )

print(f"\nBuilding GW network with detectors: {list(detector_settings.keys())}")
t0 = time.time()
gw_network = GWNetwork(detector_settings, injection_parameters=None)
print(f"Network built in {time.time() - t0:.2f} s")

batched_detector = gw_network.batched_detector
log_likelihood = partial(log_likelihood_det, detector_list=batched_detector)

## 6 — Priors

The prior choices below match Section IV of arXiv:2210.15684 exactly.

**Corrections vs. `gw170817_pe_sharpy_original.py`**:

1. **Distance**: The paper uses a prior uniform in luminosity volume, $p(d_L) \propto d_L^2$. In the log-distance parameterisation the Jacobian gives an extra factor, so the log-prior contribution is `3 * params[2]` (i.e. $3\ln d_L$) instead of `0.0`.
2. **Inclination**: Isotropic $\Rightarrow$ sine prior, $p(\theta_{JN}) \propto \sin\theta$. In log-prior: `log|sin(theta_jn)|`. Clamped to avoid `log(0)` at the poles.
3. **Declination**: Isotropic $\Rightarrow$ cosine prior, $p(\delta) \propto \cos\delta$. In log-prior: `log|cos(dec)|`.
4. **Chirp mass prior**: Uniform `[1.18, 1.21]` M$_\odot$ — unchanged, already correct.
5. **Prior bounds**: log-distance bounds remain `[log(10), log(100)]` as before, but the prior *weight* is now volumetric.

In [ ]:
# Prior bounds — 13-parameter vector
prior_bounds = jnp.array([
    [0.0,              2 * jnp.pi],       # [0]  ra
    [-jnp.pi / 2,      jnp.pi / 2],       # [1]  dec
    [jnp.log(10.0),    jnp.log(100.0)],   # [2]  log(d_L/Mpc) → 10–100 Mpc
    [0.0,              jnp.pi],            # [3]  theta_jn (inclination)
    [0.0,              2 * jnp.pi],        # [4]  phic (coalescence phase)
    [0.0,              jnp.pi],            # [5]  psi (polarisation)
    [1.18,             1.21],              # [6]  chirp mass M_sun
    [0.5,              1.0],               # [7]  mass ratio q
    [-0.1,             0.1],               # [8]  tc (rel. to trigger, s)
    [-0.05,            0.05],              # [9]  chi1
    [-0.05,            0.05],              # [10] chi2
    [0.0,              5000.0],            # [11] lambda_1
    [0.0,              5000.0],            # [12] lambda_2
])

# 1 = periodic boundary,  0 = reflective boundary
boundary_conditions = jnp.array([1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0])

parameter_names = [
    "ra", "dec", "log_distance", "theta_jn", "phic", "psi",
    "chirp_mass", "mass_ratio", "tc", "chi1", "chi2",
    "lambda_1", "lambda_2",
]


def prior(params):
    """
    Log-prior matching the paper (arXiv:2210.15684 §IV):
      - Uniform in comoving volume  → p(d_L) ∝ d_L²  → log-prior += 2*log(d_L) = 2*params[2]
        (note: in the log-distance parameterisation the Jacobian gives an extra factor,
         so the correct log-prior contribution from d_L is 3*params[2] - const,
         but since SHARPy's sampler only needs differences, we use 3*params[2])
      - Isotropic inclination       → p(θ_JN) ∝ sin θ  → log-prior += log|sin(theta_jn)|
      - Isotropic declination       → p(δ) ∝ cos δ     → log-prior += log|cos(dec)|
    """
    _eps = 1e-10  # clamp threshold to avoid log(0) at poles
    log_p = 0.0
    # Distance: uniform in volume (p ∝ d_L^2, Jacobian d(log d)/d(d_L) → 3 * log_d)
    log_p += 3.0 * params[2]
    # Inclination: isotropic (sine prior)
    sin_theta = jnp.abs(jnp.sin(params[3]))
    log_p += jnp.log(jnp.where(sin_theta > _eps, sin_theta, _eps))
    # Declination: isotropic (cosine prior)
    cos_dec = jnp.abs(jnp.cos(params[1]))
    log_p += jnp.log(jnp.where(cos_dec > _eps, cos_dec, _eps))
    return log_p


print("Priors defined.")
print(f"Parameter names: {parameter_names}")

## 7 — Sampler settings

SHARPy uses a Sequential Monte Carlo (SMC) sampler. The hyperparameters below are chosen to match the paper:

- `number_of_particles = 1000` matches the paper's `nlive ≈ 1000` in the dynesty run.
- `alpha = 0.95` and `step_size = 0.3` are SHARPy defaults that provide good mixing.

In [ ]:
NUMBER_OF_PARTICLES = 1000   # matches ~nlive=1000 in paper's dynesty run
STEP_SIZE           = 0.3
ALPHA               = 0.95
SEED                = 42

print(f"Sampler: SHARPy SMC")
print(f"  particles : {NUMBER_OF_PARTICLES}")
print(f"  alpha     : {ALPHA}")
print(f"  step_size : {STEP_SIZE}")
print(f"  seed      : {SEED}")

## 8 — Run the sampler

Run the SHARPy SMC sampler. This may take several hours on a single CPU core. The output is saved to `OUTDIR`.

In [ ]:
print(f"Starting SHARPy SMC sampler ({NUMBER_OF_PARTICLES} particles)...")
start = time.time()

result_dict = run_sharpy(
    log_likelihood,
    prior,
    prior_bounds,
    boundary_conditions,
    ALPHA,
    NUMBER_OF_PARTICLES,
    STEP_SIZE,
    jax.random.PRNGKey(SEED),
    folder=OUTDIR,
    label=LABEL,
)

elapsed = time.time() - start
print(f"\nSampling completed in {elapsed:.1f} s  ({elapsed/3600:.2f} h)")

samples   = result_dict["posterior_samples"]
logZ      = result_dict["logZ"]
dlogZ     = result_dict["dlogZ"]
print(f"log Z = {logZ:.2f} ± {dlogZ:.2f}")
print(f"Posterior samples shape: {samples.shape}")

## 9 — Save results

Save posterior samples and Bayesian evidence to a `.npz` file for later analysis and plotting.

In [ ]:
save_path = os.path.join(OUTDIR, f"{LABEL}_samples.npz")
np.savez(
    save_path,
    samples=np.array(samples),
    logZ=logZ,
    dlogZ=dlogZ,
    parameter_names=parameter_names,
)
print(f"Results saved to {save_path}")

## 10 — Corner plot

Plot the posterior corner plot for all 13 parameters.

In [ ]:
try:
    import corner
    fig = corner.corner(
        np.array(samples),
        labels=parameter_names,
        show_titles=True,
        title_kwargs={"fontsize": 10},
        label_kwargs={"fontsize": 10},
    )
    fig.suptitle("GW170817 — SHARPy + mlgw_bns_jax (paper-equivalent)", y=1.01)
    plot_path = os.path.join(OUTDIR, f"{LABEL}_corner.png")
    fig.savefig(plot_path, dpi=150, bbox_inches="tight")
    print(f"Corner plot saved to {plot_path}")
    import matplotlib.pyplot as plt
    plt.show()
except ImportError:
    print("Install corner with: pip install corner")

## 11 — 1D marginal posteriors for key parameters

Quick 1D marginals for the 5 most physically interesting parameters: chirp mass, mass ratio, $\Lambda_1$, $\Lambda_2$, and log-luminosity distance.

In [ ]:
import matplotlib.pyplot as plt

samples_np = np.array(samples)

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
fig.suptitle("Key 1D marginals — GW170817 SHARPy PE (paper-equivalent)", fontsize=12)

plot_params = [
    (6,  r"$\mathcal{M}_c\;[M_\odot]$"),
    (7,  r"$q$"),
    (11, r"$\Lambda_1$"),
    (12, r"$\Lambda_2$"),
    (2,  r"$\ln d_L$"),
]

for ax, (idx, label) in zip(axes, plot_params):
    ax.hist(samples_np[:, idx], bins=40, density=True, color="steelblue", alpha=0.7)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel("p.d.f.", fontsize=9)
    median = np.median(samples_np[:, idx])
    ax.axvline(median, color="crimson", ls="--", label=f"median={median:.3g}")
    ax.legend(fontsize=8)

plt.tight_layout()
marginals_path = os.path.join(OUTDIR, f"{LABEL}_marginals.png")
fig.savefig(marginals_path, dpi=150, bbox_inches="tight")
print(f"Marginals saved to {marginals_path}")
plt.show()